# Data Cleaning

This notebook takes in the source data file and aplies the following data cleaning method:  
1.  Apply an approach to remove extreme and incorrect values from your dataset
2. select two approaches to impute missing values that are logical for such time series

It outputs the `1b_dataset_cleaned.parquet` file

In [1]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from typing import Optional

## STEP 1. Loading the data

In [2]:
DATA_DIR = Path(os.getcwd()).parent.parent / "data"
SOURCE_DATA_FILE = DATA_DIR / "dataset_mood_smartphone.csv"

print("discovered source data file:", SOURCE_DATA_FILE)

discovered source data file: /Users/jortverbeek/Desktop/VU/data mining/Assignment 1/data_mining_group_9_assignment_1/data/dataset_mood_smartphone.csv


In [3]:
df = pd.read_csv(SOURCE_DATA_FILE, index_col=0)
df["time"] = pd.to_datetime(df["time"])

## STEP 2: Fixing abnormal/extreme values

Here we apply the following operations:
1. Resolve duplicate readings (for same user, same variable and same time) - if all of duplicate entries have NaN value, we also output NaN, otherwise, we take a mean of duplicate readings.  
2. Remove impossible values for unbounded duration values (i.e. turn negative readings to NaN)  
3. Apply plausability filtering - if the duration variables have values greater than time elapsed since last reading - make them NaN  
4. IQR filtering - if the variable value is above Q3 + IQR * 3, turn it to NaN 

In [4]:
duration_variables = [
    "screen",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather",
]

In [5]:
# RESOLVING DUPLICATES

prev_shape = df.shape[0]


def resolve_duplicates(series: pd.Series) -> Optional[float]:
    """
    Resolve duplicates in a pandas Series by returning the mean of non-NA values.
    If all values are NA, return NA.
    """

    dropped_na = series.dropna()
    if dropped_na.empty:
        return np.nan
    else:
        return dropped_na.mean()


df = (
    df.groupby(["id", "time", "variable"])["value"]
    .agg(resolve_duplicates)
    .reset_index()
)

curr_shape = df.shape[0]
print(f"Collapsed total {prev_shape - curr_shape} duplicate rows")

Collapsed total 43 duplicate rows


In [6]:
# REMOVING IMPOSSIBLE VALUES

prev_count = df["value"].count()

for var in duration_variables:
    df.loc[df["variable"] == var, "value"] = df.loc[
        df["variable"] == var, "value"
    ].where(df["value"] >= 0, np.nan)

curr_count = df["value"].count()

print("Total rows found with negative values:", prev_count - curr_count)

Total rows found with negative values: 4


In [7]:
# PLAUSABILITY FILTERING

df = df.sort_values("time")
df["time_since_last_reading"] = (
    df.groupby(["id", "variable"])["time"].diff().dt.total_seconds()
)

# we fill the first entries (time wise) of each variable to positive infinity, as they can't be plausibly filtered
df["time_since_last_reading"] = df["time_since_last_reading"].fillna(np.inf)

prev_count = df["value"].count()

for var in duration_variables:
    df.loc[df["variable"] == var, "value"] = df.loc[
        df["variable"] == var, "value"
    ].where(df["value"] <= df["time_since_last_reading"], np.nan)

curr_count = df["value"].count()

print("Total rows nulled with plausability filtering:", prev_count - curr_count)

Total rows nulled with plausability filtering: 45976


In [8]:
# IQR FILTERING

prev_count = df["value"].count()


def calculate_iqr_threshold(series: pd.Series) -> float:
    """
    Calculate the upper threshold for outliers using the IQR method.
    Returns the upper threshold value.
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_threshold = q3 + 3 * iqr
    return upper_threshold


thresholds = df.groupby(["id", "variable"])["value"].transform(calculate_iqr_threshold)

is_target_var = df["variable"].isin(duration_variables)
is_over_threshold = df["value"] > thresholds

df.loc[is_target_var & is_over_threshold, "value"] = np.nan

curr_count = df["value"].count()

print("Total rows nulled with outlier filtering:", prev_count - curr_count)

Total rows nulled with outlier filtering: 14382


In [9]:
# REMAINING NULLS
for var in df["variable"].unique():
    print(
        f"Missing values for {var}:",
        df.query(f"variable == '{var}'")["value"].isna().sum(),
    )

Missing values for call: 0
Missing values for sms: 0
Missing values for circumplex.arousal: 46
Missing values for circumplex.valence: 154
Missing values for mood: 0
Missing values for appCat.other: 724
Missing values for appCat.builtin: 15943
Missing values for screen: 16985
Missing values for appCat.communication: 16333
Missing values for appCat.social: 3910
Missing values for appCat.utilities: 253
Missing values for appCat.entertainment: 4447
Missing values for appCat.travel: 403
Missing values for appCat.unknown: 171
Missing values for appCat.office: 701
Missing values for appCat.finance: 326
Missing values for activity: 0
Missing values for appCat.game: 154
Missing values for appCat.weather: 12


## STEP 3: Missing value imputation

After applying our previous methods of detecting and nulling extreme/abnormal values, we'll now use bounded linear interpolation to fill them.  
Our chosen interpolation method works as such: if a null value is within 24h of valid readings - use the timestamp of this null value to perform proper linear interpolation.  
If the null value is nowhere close to a valid reading - drop it entirely!

In [11]:
def time_limited_interpolation(df, column, threshold_hours=24):
    df = df.set_index("time")
    df = df.sort_index()

    interpolated = df[column].interpolate(method="time")

    # Identify the "Holes"
    not_null_indices = df.index[df[column].notna()]

    # Forward fill the last valid timestamp and backward fill the next valid timestamp
    last_valid_ts = pd.Series(not_null_indices, index=not_null_indices).reindex(
        df.index, method="ffill"
    )
    next_valid_ts = pd.Series(not_null_indices, index=not_null_indices).reindex(
        df.index, method="bfill"
    )

    gap_duration = next_valid_ts - last_valid_ts

    # Mask the values where the gap is too large
    # If the gap between anchors is > threshold, we want to keep it as NaN
    mask = gap_duration > pd.Timedelta(hours=threshold_hours)

    # Apply the mask: keep interpolated values only for small gaps
    return interpolated.where(~mask, np.nan)

In [12]:
df = (
    df.groupby(["id", "variable"])
    .apply(lambda x: time_limited_interpolation(x, "value", threshold_hours=24))
    .reset_index()
)

In [12]:
# REMAINING NULLS
for var in df["variable"].unique():
    print(
        f"Missing values for {var}:",
        df.query(f"variable == '{var}'")["value"].isna().sum(),
    )

Missing values for activity: 0
Missing values for appCat.builtin: 31
Missing values for appCat.communication: 12
Missing values for appCat.entertainment: 73
Missing values for appCat.finance: 47
Missing values for appCat.game: 18
Missing values for appCat.office: 23
Missing values for appCat.other: 36
Missing values for appCat.social: 50
Missing values for appCat.travel: 47
Missing values for appCat.unknown: 43
Missing values for appCat.utilities: 40
Missing values for appCat.weather: 1
Missing values for call: 0
Missing values for circumplex.arousal: 0
Missing values for circumplex.valence: 17
Missing values for mood: 0
Missing values for screen: 18
Missing values for sms: 0


In [13]:
# DROP THE REMAINING NULLS
df = df.dropna(subset=["value"])

## STEP 4: exporting cleaned data

In [14]:
OUTPUT_DATA_FILE = DATA_DIR / "1b_dataset_cleaned.parquet"
df.to_parquet(OUTPUT_DATA_FILE, index=False)